# Debug the Tiny Transformer

You've inherited a small causal language model from a teammate. They claim it "almost works" but it crashes during training, and even when patched superficially the loss never improves. The model is a stack of **8 transformer blocks**.

The homework has two parts:

**Part 1 — Debugging (Main Part).** Find and fix every bug in `model.py` so this notebook trains the model end-to-end to ≥ 0.95 copy-task accuracy.

**Part 2 — Layer recovery (Bonus).** A teammate trained the model to convergence, then mishandled the checkpoint and the 8 blocks got shuffled. You're given the shuffled weights. Recover the correct block ordering so the model works again.

The dataset (already on disk in `data/`) contains length-17 sequences of the form

$$[a_1,\, a_2,\, \dots,\, a_8,\,\mathrm{SEP},\, a_1,\, a_2,\, \dots,\, a_8]$$

where the $a_i$ are i.i.d. uniform tokens from $\{1, \dots, 31\}$ and $\mathrm{SEP} = 0$. A correctly-trained causal LM, prompted with the prefix and SEP, must autoregressively reproduce the prefix.

## Rules

- You may change **only `model.py`**. Do not modify this notebook, the data, or any fixed hyperparameters set in the setup cell.
- Bug fixes must not change parameter **names** or **shapes** — Part 2's saved weights must remain loadable.
- No GPU. Everything must run on CPU.

## Deliverables

**Part 1 (2pts)**
- **(A)** A table listing every bug you found with: file/line, cause (one sentence), minimal fix. (There are 12 bugs.)
- **(B)** A fixed `model.py` such that this notebook reports `PASS` for Part 1.

**Part 2 (Bonus)**
- **(C)** The recovered permutation `block_order` (a list of 8 integers). The notebook will PASS only when both copy accuracy ≥ 0.95 **and** the MSE checksum against the trained model's reference output is below `1e-5`.
- **(D)** A very concise paragraph describing the procedure you used to recover the order. Brute-force search is **NOT** acceptable. Think analytical / algorithmic. Hint : What quantity might be predictive of the depth in the NN ?

## Submission

Add **markdown cells inside this notebook** for the written deliverables:
- For **(A)**, add a markdown cell after the Part 1 grading cell containing your bug table.
- For **(D)**, add a markdown cell after the Part 2 grading cell describing your approach.

If you are not comfortable with Markdown syntax, ask ChatGPT or Claude. 

When done, **run the notebook top-to-bottom** so every cell shows its output, save the notebook, then zip exactly two files and submit:

```
submission.zip
├── model.py            
└── assignment.ipynb  
```

Do **not** include the `data/` folder, the conda env, or any other files.

# Part 1: Debugging

## Setup (do not modify)

In [ ]:
import torch
import matplotlib.pyplot as plt
from pathlib import Path

from model import TinyDecoder, make_optimizer, train_step

SEED = 0
N_STEPS = 500
BATCH_SIZE = 32
VOCAB_SIZE = 32
PREFIX_LEN = 8
ACCURACY_THRESHOLD = 0.95
MSE_THRESHOLD = 1e-5

torch.manual_seed(SEED)

data_dir = Path('data')
train_data = torch.load(data_dir / 'train.pt')
test_data  = torch.load(data_dir / 'test.pt')
print(f'train: {tuple(train_data.shape)}   test: {tuple(test_data.shape)}')

## Training loop (do not modify)

In [ ]:
model = TinyDecoder(vocab_size=VOCAB_SIZE)
opt = make_optimizer(model)

sampler = torch.Generator().manual_seed(42)
losses = []
for step in range(N_STEPS):
    idx = torch.randint(0, len(train_data), (BATCH_SIZE,), generator=sampler)
    loss = train_step(model, opt, train_data[idx])
    losses.append(loss)
    if step % 50 == 0:
        print(f'step {step:4d}   loss {loss:.4f}')
print(f'final training loss: {losses[-1]:.4f}')

In [ ]:
plt.plot(losses)
plt.xlabel('step')
plt.ylabel('loss')
plt.title('Training loss')
plt.legend()
plt.show()

## Evaluation (do not modify)

In [ ]:
@torch.no_grad()
def copy_accuracy(model, data, prefix_len=PREFIX_LEN):
    model.eval()
    out = data[:, : prefix_len + 1].clone()
    for _ in range(prefix_len):
        logits = model(out)
        nxt = logits[:, -1].argmax(-1)
        out = torch.cat([out, nxt.unsqueeze(1)], dim=1)
    pred = out[:, prefix_len + 1 :]
    target = data[:, prefix_len + 1 :]
    return (pred == target).float().mean().item()

acc = copy_accuracy(model, test_data)
print(f'Part 1 test copy accuracy: {acc:.4f}')
print(f'threshold:                 {ACCURACY_THRESHOLD:.4f}')
part1_pass = acc >= ACCURACY_THRESHOLD
print('Part 1: PASS' if part1_pass else 'Part 1: FAIL')

In [ ]:
@torch.no_grad()
def generate(model, prompt, n):
    model.eval()
    out = prompt.clone()
    for _ in range(n):
        logits = model(out)
        nxt = logits[:, -1].argmax(-1, keepdim=True)
        out = torch.cat([out, nxt], dim=1)
    return out

for i in range(5):
    seq = test_data[i:i+1]
    prompt = seq[:, : PREFIX_LEN + 1]
    gen = generate(model, prompt, PREFIX_LEN)
    print(f'prefix:    {seq[0, :PREFIX_LEN].tolist()}')
    print(f'predicted: {gen[0, PREFIX_LEN + 1:].tolist()}')
    print()

# Part 2: Recover the shuffled layers

A colleague trained the **same architecture** to convergence on this task, saved the weights, then accidentally permuted the 8 transformer blocks before sending you the checkpoint. The architecture is identical to yours after fixing Part 1, only the layer ordering is scrambled.

Your job: discover the permutation `p` such that setting

```python
model.block_order = p
```

after loading the shuffled weights recovers the original trained model. Concretely, the model should execute its blocks in the order `p[0], p[1], …, p[7]`.

Two signals are available to you:
1. **Copy accuracy on the test set** 
2. **An MSE checksum** against a fixed `(x_fixed, y_fixed)` pair captured from the original trained model.

## Load shuffled weights and checksum (do not modify)

In [ ]:
shuffled_sd = torch.load(data_dir / 'shuffled.pt')
x_fixed = torch.load(data_dir / 'checksum_x.pt')
y_fixed = torch.load(data_dir / 'checksum_y.pt')

shuffled_model = TinyDecoder(vocab_size=VOCAB_SIZE)
shuffled_model.load_state_dict(shuffled_sd)
shuffled_model.eval()

print(f'loaded {len(shuffled_sd)} state_dict entries')
print(f'checksum input shape:  {tuple(x_fixed.shape)}')
print(f'checksum output shape: {tuple(y_fixed.shape)}')

# the shuffled model should perform poorly.
acc_default = copy_accuracy(shuffled_model, test_data)
print(f'\nshuffled model accuracy with default block_order = [0..7]: {acc_default:.4f}')

## Your turn: search for the permutation

Add your own cells below to explore !!

## Final answer (fill in)

REPLACE with your recovered permutation of [0..7]

In [ ]:
recovered_permutation = [0, 1, 2, 3, 4, 5, 6, 7]  

## Grading cell (do not modify)

In [ ]:
assert sorted(recovered_permutation) == list(range(8)), 'must be a permutation of [0..7]'

shuffled_model.block_order = list(recovered_permutation)
shuffled_model.eval()

acc2 = copy_accuracy(shuffled_model, test_data)
with torch.no_grad():
    y_check = shuffled_model(x_fixed)
mse = (y_check - y_fixed).pow(2).mean().item()

print(f'recovered permutation:  {recovered_permutation}')
print(f'test copy accuracy:     {acc2:.4f}   (need >= {ACCURACY_THRESHOLD})')
print(f'MSE vs checksum:        {mse:.3e}   (need <  {MSE_THRESHOLD})')

part2_pass = (acc2 >= ACCURACY_THRESHOLD) and (mse < MSE_THRESHOLD)
print()
print('Part 1:', 'PASS' if part1_pass else 'FAIL')
print('Part 2:', 'PASS' if part2_pass else 'FAIL')
print('Overall:', 'PASS' if (part1_pass and part2_pass) else 'FAIL')